# Introduction to Pydantic AI

Pydantic AI is a Python agent framework designed to make it less painful to build production grade applications with Generative AI.

It's built by the team behind Pydantic, which is widely used for data validation in Python (and heavily used by frameworks like FastAPI).

```mermaid
flowchart LR
    User -->|Prompt| Agent["Pydantic AI Agent"]
    Agent -->|System Prompt| LLM["LLM (OpenAI, etc.)"]
    LLM -->|Response| Agent
    Agent -->|Validated Output| User
```

Key features:
- **Model-agnostic**: Works with OpenAI, Anthropic, Gemini, Groq, and more
- **Type-safe**: Leverages Pydantic for input/output validation
- **Pythonic**: Uses standard Python patterns (no complex abstractions)
- **Observability**: Built-in support for Logfire and OpenTelemetry

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import os

from dotenv import load_dotenv
from pydantic_ai import Agent

load_dotenv()

## Basics

The core building block in Pydantic AI is the `Agent`. You specify a model and a system prompt, then call `run_sync` (or `run` for async) to get a response.

In [ ]:
agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're Jose. You're a helpful assistant that replies in haikus.",
)

response = agent.run_sync("What is the color of the sky?")
print(response.output)

## Prompt templates (Using Jinja2)

In [ ]:
from jinja2 import Template


def load_prompt(prompt_filename, variables=None):
    variables = variables or {}
    with open(prompt_filename, "r") as f:
        template = Template(f.read())
        return template.render(**variables)


system_prompt = load_prompt("assets/system_prompt.jinja2")

questions = ["What is the color of the sky?", "What is the color of the grass?"]
user_prompt = load_prompt(
    "assets/user_prompt.jinja2",
    variables={"questions": questions},
)

agent = Agent(
    "openai:gpt-5-nano",
    system_prompt=system_prompt,
)

response = agent.run_sync(user_prompt)
print(response.output)

## Streaming

Pydantic AI supports streaming responses out of the box.

In [ ]:
agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're Jose. You're a helpful assistant that replies in haikus.",
)


async def stream_response():
    async with agent.run_stream("What is the color of the sky?") as response:
        async for chunk in response.stream_text(delta=True):
            print(chunk, end="", flush=True)


await stream_response()

## Multimodality

In [ ]:
from pydantic_ai import BinaryContent

agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're Jose. You're a helpful assistant who always replies in haikus.",
)

with open("assets/dog_image.png", "rb") as f:
    image_data = f.read()

response = agent.run_sync(
    [
        "Describe this image:",
        BinaryContent(data=image_data, media_type="image/png"),
    ]
)
print(response.output)

## Observability with Logfire

Pydantic AI is compatible with OpenTelemetry (OTel), and integrates natively with [Logfire](https://logfire.pydantic.dev/).

To enable tracking, create a project in Logfire, generate a `Write token` and add it to the `.env` file:

In [ ]:
import logfire

logfire.configure()
logfire.instrument_pydantic_ai()

In [ ]:
agent = Agent(
    "openai:gpt-5-nano",
    system_prompt="You're Jose. You're a helpful assistant that replies in haikus.",
)

response = agent.run_sync("What is the color of the sky?")
print(response.output)

# Exercise

Create a Pydantic AI agent that takes a topic and a style as input, and generates a response in that style (e.g., haiku, limerick, sonnet).